# 1. Dataset Overview

## 1.1 Structure and Basic Characteristics 

In [3]:
import pandas as pd

df = pd.read_json("../data/raw_credit_applications.json")
df.head()

,_id,applicant_info,financials,spending_behavior,decision,processing_timestamp,loan_purpose,notes
0,app_200,"{'full_name': 'Jerry Smith', 'email': 'jerry.s...","{'annual_income': 73000, 'credit_history_month...","[{'category': 'Shopping', 'amount': 480}, {'ca...","{'loan_approved': False, 'rejection_reason': '...",2024-01-15T00:00:00Z,NaN,NaN
1,app_037,"{'full_name': 'Brandon Walker', 'email': 'bran...","{'annual_income': 78000, 'credit_history_month...","[{'category': 'Rent', 'amount': 608}, {'catego...","{'loan_approved': False, 'rejection_reason': '...",NaN,NaN,NaN
2,app_215,"{'full_name': 'Scott Moore', 'email': 'scott.m...","{'annual_income': 61000, 'credit_history_month...","[{'category': 'Rent', 'amount': 109}]","{'loan_approved': True, 'interest_rate': 3.7, ...",NaN,vacation,NaN
3,app_024,"{'full_name': 'Thomas Lee', 'email': 'thomas.l...","{'annual_income': 103000, 'credit_history_mont...","[{'category': 'Fitness', 'amount': 575}]","{'loan_approved': True, 'interest_rate': 4.3, ...",NaN,NaN,NaN
4,app_184,"{'full_name': 'Brian Rodriguez', 'email': 'bri...","{'annual_income': 57000, 'credit_history_month...","[{'category': 'Entertainment', 'amount': 463}]","{'loan_approved': False, 'rejection_reason': '...",2024-01-15T00:00:00Z,NaN,NaN


In [4]:
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns)
print("\nData types:")
print(df.dtypes)

Shape: (502, 8)

Columns:
Index(['_id', 'applicant_info', 'financials', 'spending_behavior', 'decision',
       'processing_timestamp', 'loan_purpose', 'notes'],
      dtype='object')

Data types:
_id                     object
applicant_info          object
financials              object
spending_behavior       object
decision                object
processing_timestamp    object
loan_purpose            object
notes                   object
dtype: object


### Observations: 

The dataset contains 502 credit applications and 8 top-level fields.

The structure is semi-structured (nested JSON), meaning several fields contain embedded objects and arrays. This increases the risk of structural inconsistencies and hidden data quality issues.

All columns are currently loaded as type "object", meaning further normalization is required for proper auditing and validation.

# 2. Data Quality Assessment 

We assess the dataset using the six data quality dimensions defined in the course:

1. Completeness  
2. Uniqueness  
3. Validity  
4. Consistency  
5. Timeliness  
6. Accuracy  

### 2.1 Completeness

Completeness evaluates whether required data fields are present and populated.

Given the nested JSON structure, completeness must be assessed both at:
- Top-level fields
- Nested object fields (applicant_info, financials, decision)

In [5]:
df.isnull().sum()

_id                       0
applicant_info            0
financials                0
spending_behavior         0
decision                  0
processing_timestamp    440
loan_purpose            452
notes                   500
dtype: int64

In [6]:
applicant_df = pd.json_normalize(df["applicant_info"])
financials_df = pd.json_normalize(df["financials"])
decision_df = pd.json_normalize(df["decision"])

In [7]:
print("Applicant missing:")
print(applicant_df.isnull().sum())

print("\nFinancials missing:")
print(financials_df.isnull().sum())

print("\nDecision missing:")
print(decision_df.isnull().sum())

Applicant missing:
full_name        0
email            0
ssn              5
ip_address       5
gender           1
date_of_birth    1
zip_code         1
dtype: int64

Financials missing:
annual_income              5
credit_history_months      0
debt_to_income             0
savings_balance            0
annual_salary            497
dtype: int64

Decision missing:
loan_approved         0
rejection_reason    292
interest_rate       210
approved_amount     210
dtype: int64


In [8]:
total = len(df)

print("Top-level missing percentage:")
print((df.isnull().sum() / total * 100).round(2))

print("\nApplicant missing percentage:")
print((applicant_df.isnull().sum() / total * 100).round(2))

print("\nFinancials missing percentage:")
print((financials_df.isnull().sum() / total * 100).round(2))

print("\nDecision missing percentage:")
print((decision_df.isnull().sum() / total * 100).round(2))

Top-level missing percentage:
_id                      0.00
applicant_info           0.00
financials               0.00
spending_behavior        0.00
decision                 0.00
processing_timestamp    87.65
loan_purpose            90.04
notes                   99.60
dtype: float64

Applicant missing percentage:
full_name        0.0
email            0.0
ssn              1.0
ip_address       1.0
gender           0.2
date_of_birth    0.2
zip_code         0.2
dtype: float64

Financials missing percentage:
annual_income             1.0
credit_history_months     0.0
debt_to_income            0.0
savings_balance           0.0
annual_salary            99.0
dtype: float64

Decision missing percentage:
loan_approved        0.00
rejection_reason    58.17
interest_rate       41.83
approved_amount     41.83
dtype: float64


### Findings – Completeness

Completeness was assessed at both the top-level and nested-object level due to the semi-structured JSON design of the dataset.

At the top level, three attributes show critical missingness:
notes (99.6%), loan_purpose (90.0%), and processing_timestamp (87.7%).
The high absence of processing_timestamp raises governance and auditability concerns, as timestamps are typically required in financial systems.

Within nested objects, applicant and core financial attributes are largely complete (below 1% missing in most cases). However, annual_salary is missing in nearly all records (98.9%), indicating a likely schema inconsistency rather than random data loss.

Decision-related attributes (rejection_reason, interest_rate, approved_amount) display conditional missingness, as their presence depends on loan approval status. This suggests the need for cross-field validation rather than direct completeness correction.

Overall, completeness risks are concentrated in metadata fields and schema design rather than in primary applicant data.

### 2.2 Uniqueness

Uniqueness evaluates whether records that should be distinct are not duplicated.

In [9]:
print("Duplicate _id count:")
print(df["_id"].duplicated().sum())

Duplicate _id count:
2


In [12]:
print("Duplicate applicant SSN:")
print(applicant_df["ssn"].duplicated().sum())

Duplicate applicant SSN:
7


In [13]:
print("Duplicate email:")
print(applicant_df["email"].duplicated().sum())

Duplicate email:
8


In [ ]:
df[df["_id"].duplicated(keep=False)].  #duplicates

,_id,applicant_info,financials,spending_behavior,decision,processing_timestamp,loan_purpose,notes
8,app_042,"{'full_name': 'Joseph Lopez', 'email': 'joseph...","{'annual_income': 69000, 'credit_history_month...","[{'category': 'Insurance', 'amount': 153}, {'c...","{'loan_approved': False, 'rejection_reason': '...",NaN,NaN,NaN
354,app_042,"{'full_name': 'Joseph Lopez', 'email': 'joseph...","{'annual_income': 69000, 'credit_history_month...","[{'category': 'Insurance', 'amount': 153}, {'c...","{'loan_approved': False, 'rejection_reason': '...",NaN,NaN,RESUBMISSION
383,app_001,"{'full_name': 'Stephanie Nguyen', 'email': 'st...","{'annual_income': 102000, 'credit_history_mont...","[{'category': 'Fitness', 'amount': 576}]","{'loan_approved': False, 'rejection_reason': '...",NaN,NaN,NaN
455,app_001,"{'full_name': 'Stephanie Nguyen', 'email': 'st...","{'annual_income': 102000, 'credit_history_mont...","[{'category': 'Fitness', 'amount': 576}]","{'loan_approved': False, 'rejection_reason': '...",NaN,NaN,DUPLICATE_ENTRY_ERROR


### Findings – Uniqueness

Uniqueness was assessed by verifying that identifiers expected to be unique do not contain duplicates.

Two duplicate _id values were detected. Since _id represents the application identifier, duplicates indicate a structural data integrity issue in the ingestion or storage pipeline.

Seven duplicate SSNs and eight duplicate emails were also identified. While these may reflect repeat applications by the same individual, they may also indicate potential fraud risk or insufficient identity validation controls.

Overall, uniqueness issues are primarily structural at the application ID level and require remediation at the data ingestion stage.